# OCR Comparison: Tesseract, EasyOCR, and PaddleOCR

Target image: `Icdar2013\\Challenge2_Test_Task12_Images\\img_1.jpg`

## Shared: Read Ground Truth

**Syntax:** `Path.read_text(encoding, errors)`

**Intro:** Load ground truth text from file for comparison. Uses `pathlib.Path` to read UTF-8 encoded text files.

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pytesseract
# Set this if Tesseract is not in PATH
pytesseract.pytesseract.tesseract_cmd = r"C:\\Program Files\\Tesseract-OCR\\tesseract.exe"



## 1. Tesseract OCR on the Same Image

### 1) Initialize Tesseract

**Syntax:** `pytesseract.pytesseract.tesseract_cmd = r"path\\to\\tesseract.exe"`

**Intro:** Configure the path to Tesseract OCR executable. Required so PyTesseract knows where to find the OCR engine on your system.

### 2) Read image

**Syntax:** `cv2.imread(filename) -> numpy.ndarray`

**Intro:** Load image from disk using OpenCV. Returns BGR image as NumPy array. Check `.shape` to verify dimensions (height, width, channels).

### 3) Apply OCR

**Syntax:** `pytesseract.image_to_string(image, lang="eng", config="--oem 3 --psm 6") -> str`

**Intro:** Extract text from image using Tesseract OCR. Returns detected text as string. Can use optional config flags for engine mode (oem) and page segmentation mode (psm).

### 4) Calculate CER and WER

**Syntax:** `levenshtein_distance(seq1, seq2) -> int` | CER = distance / ref_chars | WER = distance / ref_words

**Intro:** Compute edit distance between ground truth and predicted text. CER (Character Error Rate) and WER (Word Error Rate) measure OCR accuracy. Lower is better.

In [ ]:
def levenshtein_distance(seq1, seq2):
    n, m = len(seq1), len(seq2)
    if n == 0:
        return m
    if m == 0:
        return n
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        dp[i][0] = i
    for j in range(m + 1):
        dp[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = 0 if seq1[i - 1] == seq2[j - 1] else 1
            dp[i][j] = min(
                dp[i - 1][j] + 1,
                dp[i][j - 1] + 1,
                dp[i - 1][j - 1] + cost
            )
    return dp[n][m]


def compute_cer_wer(gt_text, pred_text):
    gt_norm = gt_text.strip()
    pred_norm = pred_text.strip()

    char_dist = levenshtein_distance(list(gt_norm), list(pred_norm))
    cer = char_dist / max(1, len(gt_norm))

    gt_words = gt_norm.split()
    pred_words = pred_norm.split()
    word_dist = levenshtein_distance(gt_words, pred_words)
    wer = word_dist / max(1, len(gt_words))

    return {
        "cer": cer,
        "wer": wer,
        "char_distance": char_dist,
        "word_distance": word_dist,
        "gt_num_chars": len(gt_norm),
        "pred_num_chars": len(pred_norm),
        "gt_num_words": len(gt_words),
        "pred_num_words": len(pred_words),
    }


def show_metrics(name, gt_text, pred_text):
    metrics = compute_cer_wer(gt_text, pred_text)
    print(f"{name} metrics:")
    print(metrics)
    return metrics




## 2. EasyOCR on the Same Image

**Syntax:** `easyocr.Reader(["en"]).readtext(image_path)`

**Intro:** Run EasyOCR on the same input image, join detected text segments, and compute CER/WER with the shared helper functions.

### Functions used in the next EasyOCR code cells

**Syntax:** `easyocr.Reader(languages, gpu=False)`

**Intro:** Creates the OCR reader object and loads EasyOCR models for the selected language set.

**Syntax:** `reader.readtext(image, detail=1, paragraph=False)`

**Intro:** Runs OCR on the image and returns detection results (boxes, text, confidence).

**Syntax:** `" ".join(iterable)`

**Intro:** Combines recognized text segments into one string for metric calculation.

**Syntax:** `show_metrics(name, gt_text, pred_text)`

**Intro:** Calls `compute_cer_wer(...)`, then prints CER/WER and edit distances.

## 3. PaddleOCR on the Same Image

**Syntax:** `PaddleOCR(...).predict(image_path)`

**Intro:** Run PaddleOCR on the same image, flatten recognized text lines, and evaluate with the same CER/WER function for side-by-side comparison.

### Functions used in the next PaddleOCR code cells

**Syntax:** `PaddleOCR(use_textline_orientation=True, lang="en")`

**Intro:** Initializes PaddleOCR with English recognition and text-line orientation handling.

**Syntax:** `ocr.predict(image_path)`

**Intro:** Runs OCR and returns page-level prediction objects that include recognized texts.

**Syntax:** `page.get("rec_texts", [])`

**Intro:** Reads recognized text lines safely from each page prediction.

**Syntax:** `" ".join(iterable)`

**Intro:** Merges recognized lines into one string for metric calculation.

**Syntax:** `show_metrics(name, gt_text, pred_text)`

**Intro:** Uses `compute_cer_wer(...)` and prints CER/WER with distances.

In [ ]:
# pip install paddleocr paddlepaddle



## 4. Save Outputs (All Methods)

**Syntax:** `Path.mkdir(exist_ok=True)` | `Path.write_text(content, encoding="utf-8")`

**Intro:** Create output directory and save OCR text outputs for all three methods (Tesseract, EasyOCR, PaddleOCR).